### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="santander_customer_transaction_prediction",
    dataset_year="2019",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/c/santander-customer-transaction-prediction",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c santander-customer-transaction-prediction -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/santander_customer_transaction_prediction && mv train.csv local-data-warehouse/santander_customer_transaction_prediction/
""",
    # References
    academic_reference_bibtex=r"""@misc{Piedra2019SantanderCustomerTransactionPrediction,
  author = {Mercedes Piedra and Sohier Dane and Soraya Jimenez},
  title  = {Santander Customer Transaction Prediction},
  year   = {2019},
  howpublished = {\url{https://kaggle.com/competitions/santander-customer-transaction-prediction}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Piedra2019SantanderCustomerTransactionPrediction",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- The data seems to be generated or created for the competition. Most features are perfectly normally distributed.
- Kaggle experts found various uniqueness-based features and had to filter fake samples from the test data. We apply the uniqueness-based features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="target",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

# Follow #1 solution https://www.kaggle.com/code/fl2ooo/create-data
# - We do not create the feature that is used for / requires test-time adaption for the fake test samples (that do not exist in the train data).
orig = [f'var_{i}' for i in range(200)]
has_one = [f'var_{i}_has_one' for i in range(200)]
has_zero = [f'var_{i}_has_zero' for i in range(200)]
target = task_mold.target_column_name

for f in orig:
    df[f + '_has_one'] = 0
    df[f + '_has_zero'] = 0
    f_1 = df.loc[df[target] == 1, f].value_counts()

    f_1_1 = set(f_1.index[f_1 > 1])
    f_0_1 = set(f_1.index[f_1 > 0])

    f_0 = df.loc[df[target] == 0, f].value_counts()
    f_0_0 = set(f_0.index[f_0 > 1])
    f_1_0 = set(f_0.index[f_0 > 0])

    df.loc[df[target] == 1, f + '_has_one'] = df.loc[df[target] == 1, f].isin(f_1_1).astype(int)
    df.loc[df[target] == 0, f + '_has_one'] = df.loc[df[target] == 0, f].isin(f_0_1).astype(int)
    df.loc[df[target] == 1, f + '_has_zero'] = df.loc[df[target] == 1, f].isin(f_1_0).astype(int)
    df.loc[df[target] == 0, f + '_has_zero'] = df.loc[df[target] == 0, f].isin(f_0_0).astype(int)
    df = df.copy()

df.loc[:, has_one] = 2*df.loc[:, has_one].values + df.loc[:, has_zero].values

df = df.drop(columns=["ID_code"])

cat_cols = [task_mold.target_column_name] + has_one + has_zero
for col in cat_cols:
    df[col] = df[col].astype("category")

df = df.reset_index(drop=True)

Loaded data shape: (200000, 202)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 200,000
Columns: 601
Use sampling: False (sample size: 200,000)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['var_45', 'var_117', 'var_74', 'var_61', 'var_97', 'var_120', 'var_90', 'var_187', 'var_136', 'var_160']
Rows remaining as candidates after top-10 filter: 0 (of 200,000)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...


Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,var_10,var_11,var_12,var_13,var_14,var_15,var_16,var_17,var_18,var_19,var_20,var_21,var_22,var_23,var_24,var_25,var_26,var_27,var_28,var_29,var_30,var_31,var_32,var_33,var_34,var_35,var_36,var_37,var_38,var_39,var_40,var_41,var_42,var_43,var_44,var_45,var_46,var_47,var_48,var_49,var_50,var_51,var_52,var_53,var_54,var_55,var_56,var_57,var_58,var_59,var_60,var_61,var_62,var_63,var_64,var_65,var_66,var_67,var_68,var_69,var_70,var_71,var_72,var_73,var_74,var_75,var_76,var_77,var_78,var_79,var_80,var_81,var_82,var_83,var_84,var_85,var_86,var_87,var_88,var_89,var_90,var_91,var_92,var_93,var_94,var_95,var_96,var_97,var_98,var_99,var_100,var_101,var_102,var_103,var_104,var_105,var_106,var_107,var_108,var_109,var_110,var_111,var_112,var_113,var_114,var_115,var_116,var_117,var_118,var_119,var_120,var_121,var_122,var_123,var_124,var_125,var_126,var_127,var_128,var_129,var_130,var_131,var_132,var_133,var_134,var_135,var_136,var_137,var_138,var_139,var_140,var_141,var_142,var_143,var_144,var_145,var_146,var_147,var_148,var_149,var_150,var_151,var_152,var_153,var_154,var_155,var_156,var_157,var_158,var_159,var_160,var_161,var_162,var_163,var_164,var_165,var_166,var_167,var_168,var_169,var_170,var_171,var_172,var_173,var_174,var_175,var_176,var_177,var_178,var_179,var_180,var_181,var_182,var_183,var_184,var_185,var_186,var_187,var_188,var_189,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199,var_0_has_one,var_0_has_zero,var_1_has_one,var_1_has_zero,var_2_has_one,var_2_has_zero,var_3_has_one,var_3_has_zero,var_4_has_one,var_4_has_zero,var_5_has_one,var_5_has_zero,var_6_has_one,var_6_has_zero,var_7_has_one,var_7_has_zero,var_8_has_one,var_8_has_zero,var_9_has_one,var_9_has_zero,var_10_has_one,var_10_has_zero,var_11_has_one,var_11_has_zero,var_12_has_one,var_12_has_zero,var_13_has_one,var_13_has_zero,var_14_has_one,var_14_has_zero,var_15_has_one,var_15_has_zero,var_16_has_one,var_16_has_zero,var_17_has_one,var_17_has_zero,var_18_has_one,var_18_has_zero,var_19_has_one,var_19_has_zero,var_20_has_one,var_20_has_zero,var_21_has_one,var_21_has_zero,var_22_has_one,var_22_has_zero,var_23_has_one,var_23_has_zero,var_24_has_one,var_24_has_zero,var_25_has_one,var_25_has_zero,var_26_has_one,var_26_has_zero,var_27_has_one,var_27_has_zero,var_28_has_one,var_28_has_zero,var_29_has_one,var_29_has_zero,var_30_has_one,var_30_has_zero,var_31_has_one,var_31_has_zero,var_32_has_one,var_32_has_zero,var_33_has_one,var_33_has_zero,var_34_has_one,var_34_has_zero,var_35_has_one,var_35_has_zero,var_36_has_one,var_36_has_zero,var_37_has_one,var_37_has_zero,var_38_has_one,var_38_has_zero,var_39_has_one,var_39_has_zero,var_40_has_one,var_40_has_zero,var_41_has_one,var_41_has_zero,var_42_has_one,var_42_has_zero,var_43_has_one,var_43_has_zero,var_44_has_one,var_44_has_zero,var_45_has_one,var_45_has_zero,var_46_has_one,var_46_has_zero,var_47_has_one,var_47_has_zero,var_48_has_one,var_48_has_zero,var_49_has_one,var_49_has_zero,var_50_has_one,var_50_has_zero,var_51_has_one,var_51_has_zero,var_52_has_one,var_52_has_zero,var_53_has_one,var_53_has_zero,var_54_has_one,var_54_has_zero,var_55_has_one,var_55_has_zero,var_56_has_one,var_56_has_zero,var_57_has_one,var_57_has_zero,var_58_has_one,var_58_has_zero,var_59_has_one,var_59_has_zero,var_60_has_one,var_60_has_zero,var_61_has_one,var_61_has_zero,var_62_has_one,var_62_has_zero,var_63_has_one,var_63_has_zero,var_64_has_one,var_64_has_zero,var_65_has_one,var_65_has_zero,var_66_has_one,var_66_has_zero,var_67_has_one,var_67_has_zero,var_68_has_one,var_68_has_zero,var_69_has_one,var_69_has_zero,var_70_has_one,var_70_has_zero,var_71_has_one,var_71_has_zero,var_72_has_one,var_72_has_zero,var_73_has_one,var_73_has_zero,var_74_has_one,var_74_has_zero,var_75_has_one,var_75_has_zero,var_76_has_one,var_76_has_zero,var_77_has_one,var_77_has_zero,var_78_has_one,var_78_has_zero,var_79_has_one,var_79_has_zero,var_80_has_one,var_80_has_zero,var_81_has_

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,target,category,0.0,0.0,2.0,"0, 1"
1,var_0_has_one,category,0.0,0.0,4.0,"1, 0, 3, 2"
2,var_0_has_zero,category,0.0,0.0,2.0,"1, 0"
3,var_1_has_one,category,0.0,0.0,4.0,"1, 0, 3, 2"
4,var_1_has_zero,category,0.0,0.0,2.0,"1, 0"
5,var_2_has_one,category,0.0,0.0,4.0,"1, 3, 0, 2"
6,var_2_has_zero,category,0.0,0.0,2.0,"1, 0"
7,var_3_has_one,category,0.0,0.0,4.0,"1, 3, 0, 2"
8,var_3_has_zero,category,0.0,0.0,2.0,"1, 0"
9,var_4_has_one,category,0.0,0.0,4.0,"1, 3, 0, 2"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
var_0,200000.0,10.679914,3.040051,0.4084,20.3150
var_1,200000.0,-1.627622,4.050044,-15.0434,10.3768
var_2,200000.0,10.715192,2.640894,2.1171,19.3530
var_3,200000.0,6.796529,2.043319,-0.0402,13.1883
var_4,200000.0,11.078333,1.623150,5.0748,16.6714
var_5,200000.0,-5.065317,7.863267,-32.5626,17.2516
var_6,200000.0,5.408949,0.866607,2.3473,8.4477
var_7,200000.0,16.545850,3.418076,5.3497,27.6918
var_8,200000.0,0.284162,3.332634,-10.5055,10.1513
var_9,200000.0,7.567236,1.235070,3.9705,11.1506


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column           rank                     
target           1        0  179902  89.95
                 2        1   20098  10.05
var_0_has_one    1        1  126381  63.19
                 2        0   40355  20.18
                 3        3   27409  13.70
                 4        2    5855   2.93
var_0_has_zero   1        1  153790  76.90
                 2        0   46210  23.10
var_100_has_one  1        0  111407  55.70
                 2        1   76729  38.36
                 3        2    6503   3.25
                 4        3    5361   2.68
var_100_has_zero 1        0  117910  58.96
                 2        1   82090  41.04
var_101_has_one  1        1  108907  54.45
                 2        0   70054  35.03
                 3        3   13834   6.92
                 4        2    7205   3.60
var_101_has_zero 1        1  122741  61.37
                 2        0   77259  38.63
var_102_has_one  1        0  104560  52.28
                 2        1   82278  41.14
                 3        2    6761   3.38
                 4        3    6401   3.20
var_102_has_zero 1        0  111321  55.66
                 2        1   88679  44.34
var_103_has_one  1        3  181251  90.63
                 2        1   17924   8.96
                 3        0     747   0.37
                 4        2      78   0.04
var_103_has_zero 1        1  199175  99.59
                 2        0     825   0.41
var_104_has_one  1        1  128694  64.35
                 2        3   45375  22.69
                 3        0   22009  11.00
                 4        2    3922   1.96
var_104_has_zero 1        1  174069  87.03
                 2        0   25931  12.97
var_105_has_one  1        1  100409  50.20
                 2        3   91253  45.63
                 3        0    7054   3.53
                 4        2    1284   0.64
var_105_has_zero 1        1  191662  95.83
                 2        0    8338   4.17
var_106_has_one  1        1  128475  64.24
                 2        3   46279  23.14
                 3        0   21382  10.69
                 4        2    3864   1.93
var_106_has_zero 1        1  174754  87.38
                 2        0   25246  12.62
var_107_has_one  1        0   92223  46.11
                 2        1   91994  46.00
                 3        3    8759   4.38
                 4        2    7024   3.51
var_107_has_zero 1        1  100753  50.38
                 2        0   99247  49.62
var_108_has_one  1        3  184331  92.17
                 2        1   15226   7.61
                 3        0     345   0.17
                 4        2      98   0.05
var_108_has_zero 1        1  199557  99.78
                 2        0     443   0.22
var_109_has_one  1        1  116612  58.31
                 2        0   58241  29.12
                 3        3   18350   9.18
                 4        2    6797   3.40
var_109_has_zero 1        1  134962  67.48
                 2        0   65038  32.52
var_10_has_one   1        1  102283  51.14
                 2        0   78867  39.43
                 3        3   11584   5.79
                 4        2    7266   3.63
var_10_has_zero  1        1  113867  56.93
                 2        0   86133  43.07
var_110_has_one  1        1  119832  59.92
                 2        0   52487  26.24
                 3        3   21215  10.61
                 4        2    6466   3.23
var_110_has_zero 1        1  141047  70.52
                 2        0   58953  29.48
var_111_has_one  1        1  113079  56.54
                 2        3   76387  38.19
                 3        0    8827   4.41
                 4        2    1707   0.85
var_111_has_zero 1        1  189466  94.73
                 2        0   10534   5.27
var_112_has_one  1        1  128597  64.30
                 2        3   55490  27.74
                 3        0   13046   6.52
                 4        2    2867   1.43
var_112_has_zero 1        1  184087  92.04
                 2     

In [8]:
# Target Distribution
target_df

,count,pct
target,,
0,179902,89.95
1,20098,10.05


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to santander_customer_transaction_prediction/019d5dc9-73a2-7c3a-857c-8ecf5209804a


019d5dc9-73a2-7c3a-857c-8ecf5209804a
89c3ba280f38f7831e074d3c51d2cc323d275537e622d38861d069ec8c237507
